In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

SILVER_DB = "silver"
spark.sql(f"CREATE DATABASE IF NOT EXISTS {SILVER_DB}")

def apply_scd2(resource_name, bronze_load_date):
    source_table = f"bronze.{resource_name.lower()}"
    target_table = f"{SILVER_DB}.{resource_name.lower()}"

    bronze_df = spark.table(source_table).filter(F.col("bronze_load_date") == bronze_load_date)

    incoming = (
        bronze_df
        .withColumn("row_hash", F.sha2(F.to_json("resource_json"), 256))
        .dropDuplicates(["resource_id", "row_hash"])
        .withColumn("valid_from", F.current_timestamp())
        .withColumn("valid_to", F.lit(None).cast("timestamp"))
        .withColumn("is_current", F.lit(True))
    )

    if not spark.catalog.tableExists(target_table):
        incoming.write.format("delta").mode("overwrite").saveAsTable(target_table)
        print(f"[{resource_name}] Initial Silver load: {incoming.count()} rows")
        return

    silver = DeltaTable.forName(spark, target_table)

    current_silver = (
        silver.toDF().filter("is_current = true")
        .select("resource_id", F.col("row_hash").alias("existing_hash"))
    )
    changes = (
        incoming.join(current_silver, "resource_id", "left")
        .filter(
            F.col("existing_hash").isNull() |
            (F.col("row_hash") != F.col("existing_hash"))
        )
    )

    change_count = changes.count()
    if change_count == 0:
        print(f"[{resource_name}] No changes detected.")
        return

    (
        silver.alias("s")
        .merge(
            changes.select("resource_id").distinct().alias("c"),
            "s.resource_id = c.resource_id AND s.is_current = true",
        )
        .whenMatchedUpdate(set={
            "is_current": "false",
            "valid_to": "current_timestamp()",
        })
        .execute()
    )

    changes.write.format("delta").mode("append").saveAsTable(target_table)
    print(f"[{resource_name}] {change_count} new/changed versions inserted.")

from datetime import datetime
extraction_date = datetime.utcnow().strftime("%Y-%m-%d")  # matches your bronze_load_date

for resource in ["Patient", "Encounter", "Observation", "Condition"]:
    apply_scd2(resource, extraction_date)

In [0]:
%sql
-- Confirm valid_from is populated, valid_to is null, is_current is true for all rows (expected on a first run)
SELECT resource_id, valid_from, valid_to, is_current 
FROM silver.patient 
LIMIT 10